In [0]:
data = spark.read.table("`01_bronze`.raw.sales")
display(data.select(F.count("*")))

In [0]:
from pyspark.sql.functions import col, count,when

null_counts = data.select([count(when(col(c).isNull(),c)).alias(c) for c in data.columns])
display(null_counts)

In [0]:
def nullRemoval(df,column):
    return df.filter(col(column).isNotNull())
data = nullRemoval(data,'customerkey')
data = nullRemoval(data,'order_number')
null_counts_after = data.select([count(when(col(c).isNull(),c)).alias(c) for c in data.columns])
display(null_counts_after)

In [0]:
import pyspark.sql.functions as F

def standardize_date(columnName, df):
    return df.withColumn(
        "date_parsed",
        F.coalesce(
            F.try_to_date(F.col(columnName), "M/d/yyyy"),
            F.try_to_date(F.col(columnName), "M-d-yyyy"),
            F.try_to_date(F.col(columnName), "yyyy-M-d"),
            F.try_to_date(F.col(columnName), "yyyy/M/d")
        )
    ).withColumn(
        columnName,
        F.trim(F.col("date_parsed")).cast('date')
    ).drop("date_parsed")

data = standardize_date("order_date", data)
data = standardize_date("delivery_date", data)
display(data)


In [0]:
def dataTypeCast(df,column,dataType):
    return df.withColumn(column,F.trim(F.col(column)).cast(dataType))
data = dataTypeCast(data,'quantity','int')
data = dataTypeCast(data,'storekey','int')
data = dataTypeCast(data,'productkey','int')
data = dataTypeCast(data,'customerkey','int')
data = dataTypeCast(data,'line_item','int')
data = dataTypeCast(data,'currency_code','string')
data = dataTypeCast(data,'order_number','int')
display(data)

In [0]:
def fill_delivery_date_for_offline_stores(df, column):
    return df.withColumn(column, 
        F.when(
            F.col(column).isNull() & (F.col('storekey') != 0), 
            F.col('order_date')
        ).otherwise(F.col(column))
    )

data = fill_delivery_date_for_offline_stores(data, 'delivery_date')
display(data.select(F.count("*")))
